URL index: [Home](https://zzz.bwh.harvard.edu/luna-walkthrough/) | [Data](https://zzz.bwh.harvard.edu/luna-walkthrough/data/) | [S1. File QC](https://zzz.bwh.harvard.edu/luna-walkthrough/p1/) | [S2. Signal QC](https://zzz.bwh.harvard.edu/luna-walkthrough/p2) | [S3. Staging](https://zzz.bwh.harvard.edu/luna-walkthrough/p3) | [S4. Artifacts](https://zzz.bwh.harvard.edu/luna-walkthrough/p4) | [S5. Analysis](https://zzz.bwh.harvard.edu/luna-walkthrough/p5)

Notebook index: [Index](../00_index.ipynb) | [S1. File QC](../p1/00_index.ipynb) | [S2. Signal QC](../p2/00_index.ipynb) | [S3. Staging](../p3/00_index.ipynb) | [S4. Artifacts](../p4/00_index.ipynb) | [S5. Analysis](../p5/00_index.ipynb)

---

# 2.1. Working with gapped records (EDF+D)

Walkthrough URL = [https://zzz.bwh.harvard.edu/luna-walkthrough/p2/edfd/](https://zzz.bwh.harvard.edu/luna-walkthrough/p2/edfd/)

We'll initiate Luna and attach the `harm1.lst` sample list.

In [1]:
import lunapi as lp
proj = lp.proj()
proj.sample_list( '../harm1.lst' )

initiated lunapi v1.2.3 <lunapi.lunapi0.luna object at 0x123a87470> 

read 20 individuals from ../harm1.lst


## Describing EDF+D segments

The `HEADERS` command previously indicated that some recordings were potentially gapped EDF+D files.  We can recreate that here:

In [73]:
# luna harm1.lst -o out.db -s HEADERS
tbl = proj.silent_proc('HEADERS')

In [79]:
# destrat out.db +HEADERS -v EDF_TYPE
proj.table( 'HEADERS' )[[ 'ID', 'EDF_TYPE' ]]

,ID,EDF_TYPE
0,F01,EDF+D
1,F10,EDF+D
2,M10,EDF+D


We can use `SEGMENTS` to explore the structure of these three gapped EDFs (`F01`, `F10` and `M10`), although here we'll run `SEGMENTS` for all individuals to check (i.e. standard, continuous EDFs will have precisely 1 segment and 0 gaps).

In [4]:
# luna harm1.lst -o out.db id=F01,F10,M10 -s SEGMENTS
tbl = proj.silent_proc( 'SEGMENTS' ) 

In [5]:
# destrat out.db  +SEGMENTS
proj.table( 'SEGMENTS' )

,ID,NGAPS,NSEGS
0,F01,30,30
1,F02,0,1
2,F03,0,1
3,F04,0,1
4,F05,0,1
5,F06,0,1
6,F07,0,1
7,F08,0,1
8,F09,0,1
9,F10,2,3


To pull out the segment start/stop times for `F01`:

In [7]:
# destrat out.db +SEGMENTS -r SEG -v DUR_MIN START_HMS STOP_HMS -i F01
tbl = proj.table( 'SEGMENTS' , 'SEG' )
tbl.loc[ tbl.ID == 'F01' ][[ 'DUR_MIN' , 'START_HMS' , 'STOP_HMS' ]]

,DUR_MIN,START_HMS,STOP_HMS
0,9.5,22:41:30.000,22:51:00.000
20,0.5,23:40:30.000,23:41:00.000
23,4.5,23:41:30.000,23:46:00.000
25,5.0,23:46:30.000,23:51:30.000
26,3.0,23:52:00.000,23:55:00.000
27,1.0,23:56:00.000,23:57:00.000
28,1.0,23:58:00.000,23:59:00.000
29,2.5,23:59:30.000,00:02:00.000
30,15.5,00:02:30.000,00:18:00.000
31,2.0,00:29:00.000,00:31:00.000


In contrast, the other two gapped EDF+D files have fewer segments:

In [9]:
# destrat out.db +SEGMENTS -r SEG -v DUR_MIN START_HMS STOP_HMS -i F10
tbl.loc[ tbl.ID == 'F10' ][[ 'DUR_MIN' , 'START_HMS' , 'STOP_HMS' ]]

,DUR_MIN,START_HMS,STOP_HMS
9,5.016667,22:00:00.000,22:05:01.000
21,161.666667,22:05:11.200,00:46:51.200
24,294.083333,00:52:18.600,05:46:23.600


In [10]:
# destrat out.db +SEGMENTS -r SEG -v DUR_MIN START_HMS STOP_HMS -i M10
tbl.loc[ tbl.ID == 'M10' ][[ 'DUR_MIN' , 'START_HMS' , 'STOP_HMS' ]]

,DUR_MIN,START_HMS,STOP_HMS
19,8.866667,22:00:22.300,22:09:14.300
22,436.733333,22:16:18.557,05:33:02.557


Given that all EDFs had start times of `22:00:00` (10pm), this implies that `M10` has a gap at the start of the recording (i.e. as the first segment starts 22.3 seconds after this point).  We can confirm by explicitly listed the gaps:

In [11]:
# destrat out.db +SEGMENTS -r GAP -v DUR_MIN START_HMS STOP_HMS -i M10
tbl = proj.table( 'SEGMENTS' , 'GAP' )
tbl.loc[ tbl.ID == 'M10' ][[ 'DUR_MIN' , 'START_HMS' , 'STOP_HMS' ]]

,DUR_MIN,START_HMS,STOP_HMS
2,0.371667,22:00:00.000,22:00:22.300
15,7.07095,22:09:14.300,22:16:18.557


## Visualizing EDF+D segments

Here we'll use `scope()` to visualize the gapped structure of some of these EDF+D files.  For `F01` (here pulling only a subset of channels, to speed up initialization).   As described in the [main walkthrough](https://zzz.bwh.harvard.edu/luna-walkthrough/p2/edfd/#visualizing-edfd-segments), you can add the staging annotations and use the `Width` slider to control the range of the recording shown, which should make the gapped structure clear.   Here we see this individual only has N2 epochs:

In [13]:
p = proj.inst( 'F01' ) 
lp.scope( p , chs = [ 'AFZ', 'FZ', 'CZ', 'PZ', 'OZ' ] ) 

___________________________________________________________________
Processing: F01 | ../work/harm1/F01.edf
 duration 03.00.30, 10830s | time 22.00.00 - 04.58.00 | date 01.01.85

 signals: 58 (of 58) selected in an EDF+D file
  Fp1 | Fp2 | AF3 | AF4 | F7 | F5 | F3 | F1
  F2 | F4 | F6 | F8 | FT7 | FC5 | FC3 | FC1
  FC2 | FC4 | FC6 | FT8 | T7 | C5 | C3 | C1
  C2 | C4 | C6 | T8 | TP7 | CP5 | CP3 | CP1
  CP2 | CP4 | CP6 | TP8 | P7 | P5 | P3 | P1
  P2 | P4 | P6 | P8 | PO3 | PO4 | O1 | O2
  AFZ | FZ | FCZ | CZ | CPZ | PZ | POz | OZ
  FPZ | EDF Annotations
  extracting 'EDF Annotations' track from EDF+


AppLayout(children=(VBox(children=(VBox(children=(Label(value='T: 22:00:00 - 22:00:30'), HBox(children=(VBox(c…

We can do the same for `F10`:

In [14]:
p = proj.inst( 'F10' ) 
lp.scope( p , chs = [ 'AFZ', 'FZ', 'CZ', 'PZ', 'OZ' ] ) 

___________________________________________________________________
Processing: F10 | ../work/harm1/F10.edf
 duration 07.40.46, 27646s | time 22.00.00 - 05.46.23 | date 01.01.85

 signals: 58 (of 58) selected in an EDF+D file
  Fp1 | Fp2 | AF3 | AF4 | F7 | F5 | F3 | F1
  F2 | F4 | F6 | F8 | FT7 | FC5 | FC3 | FC1
  FC2 | FC4 | FC6 | FT8 | T7 | C5 | C3 | C1
  C2 | C4 | C6 | T8 | TP7 | CP5 | CP3 | CP1
  CP2 | CP4 | CP6 | TP8 | P7 | P5 | P3 | P1
  P2 | P4 | P6 | P8 | PO3 | PO4 | O1 | O2
  AFZ | FZ | FCZ | CZ | CPZ | PZ | POz | OZ
  FPZ | EDF Annotations
  extracting 'EDF Annotations' track from EDF+


AppLayout(children=(VBox(children=(VBox(children=(Label(value='T: 22:00:00 - 22:00:30'), HBox(children=(VBox(c…

Scroll to the period around `22:05` with annotations selected to see the same view as in the original walkthrough page above, i.e. showing the gap is not aligned with the 30-second epochs. 

Finally, you can do the same for `M10` - by scrolling around using `scope()` see if you can clearly identify the nature of these gaps - i.e. here staging is aligned with clock-time rather than the segment boundaries.

In [15]:
p = proj.inst( 'M10' ) 
lp.scope( p , chs = [ 'AFZ', 'FZ', 'CZ', 'PZ', 'OZ' ] ) 

___________________________________________________________________
Processing: M10 | ../work/harm1/M10.edf
 duration 07.25.36, 26736s | time 22.00.00 - 05.33.02 | date 01.01.85

 signals: 58 (of 58) selected in an EDF+D file
  Fp1 | Fp2 | AF3 | AF4 | F7 | F5 | F3 | F1
  F2 | F4 | F6 | F8 | FT7 | FC5 | FC3 | FC1
  FC2 | FC4 | FC6 | FT8 | T7 | C5 | C3 | C1
  C2 | C4 | C6 | T8 | TP7 | CP5 | CP3 | CP1
  CP2 | CP4 | CP6 | TP8 | P7 | P5 | P3 | P1
  P2 | P4 | P6 | P8 | PO3 | PO4 | O1 | O2
  AFZ | FZ | FCZ | CZ | CPZ | PZ | POz | OZ
  FPZ | EDF Annotations
  extracting 'EDF Annotations' track from EDF+


AppLayout(children=(VBox(children=(VBox(children=(Label(value='T: 22:00:00 - 22:00:30'), HBox(children=(VBox(c…

## Hypnogram statistics on EDF+D

We'll use `HYPNO` to generate hypnogram statistics just for these three gapped EDF+D files (by setting the `id` variable).  __Note that this will give a series of warnings__ about _conflicts_ as described in the main walkthrough.  

In [31]:
# luna harm1.lst -o out.db id=F01,F10,M10 -s HYPNO
proj.var( 'id', 'F01,F10,M10' )
res = proj.silent_proc( 'HYPNO' ) 

7000000 2898557000000
 conflict (>1 stage overlaps this epoch): 2928557000000 2958557000000
 conflict (>1 stage overlaps this epoch): 2988557000000 3018557000000
 conflict (>1 stage overlaps this epoch): 3048557000000 3078557000000
 conflict (>1 stage overlaps this epoch): 3078557000000 3108557000000
 conflict (>1 stage overlaps this epoch): 3138557000000 3168557000000
 conflict (>1 stage overlaps this epoch): 3168557000000 3198557000000
 conflict (>1 stage overlaps this epoch): 3198557000000 3228557000000
 conflict (>1 stage overlaps this epoch): 3228557000000 3258557000000
 conflict (>1 stage overlaps this epoch): 3258557000000 3288557000000
 conflict (>1 stage overlaps this epoch): 3288557000000 3318557000000
 conflict (>1 stage overlaps this epoch): 3948557000000 3978557000000
 conflict (>1 stage overlaps this epoch): 5688557000000 5718557000000
 conflict (>1 stage overlaps this epoch): 5718557000000 5748557000000
 conflict (>1 stage overlaps this epoch): 5808557000000 583855700000

In [32]:
# destrat out.db +HYPNO -v CONF
proj.strata()

,Command,Strata
0,HYPNO,BL
1,HYPNO,C
2,HYPNO,E
3,HYPNO,N
4,HYPNO,POST_PRE
5,HYPNO,SS


We can see the _conflict_ messages are being generated by `M10`, because of the issue described above about the non-alignment of stages to segments/epochs under default settings.

In [34]:
proj.table( 'HYPNO' , 'BL' )[[ 'ID', 'CONF' ]] 

,ID,CONF
0,F01,0
1,F10,0
2,M10,211


Looking specifically at `M10`, we'll extract the timings of the staging and the epochs, and then use a Luna command to define epochs specifically to align with the staging.  First, the staging annotations:

In [50]:
# luna harm1.lst -o out.db id=M10 -s ANNOTS annot=N1,N2,N3,R,W
p = proj.inst( 'M10' ) 
p.eval( 'ANNOTS annot=N1,N2,N3,R,W' )

___________________________________________________________________
Processing: M10 | ../work/harm1/M10.edf
 duration 07.25.36, 26736s | time 22.00.00 - 05.33.02 | date 01.01.85

 signals: 58 (of 58) selected in an EDF+D file
  Fp1 | Fp2 | AF3 | AF4 | F7 | F5 | F3 | F1
  F2 | F4 | F6 | F8 | FT7 | FC5 | FC3 | FC1
  FC2 | FC4 | FC6 | FT8 | T7 | C5 | C3 | C1
  C2 | C4 | C6 | T8 | TP7 | CP5 | CP3 | CP1
  CP2 | CP4 | CP6 | TP8 | P7 | P5 | P3 | P1
  P2 | P4 | P6 | P8 | PO3 | PO4 | O1 | O2
  AFZ | FZ | FCZ | CZ | CPZ | PZ | POz | OZ
  FPZ | EDF Annotations
  extracting 'EDF Annotations' track from EDF+
 ..................................................................
 CMD #1: ANNOTS
   options: annot=N1,N2,N3,R,W sig=*
  set epochs to default 30 seconds, 890 epochs
  keeping annotations based on any overlap with an unmasked region


,Command,Strata
0,ANNOTS,ANNOT
1,ANNOTS,ANNOT_INST
2,ANNOTS,ANNOT_INST_T1_T2


In [51]:
p.strata()

,Command,Strata
0,ANNOTS,ANNOT
1,ANNOTS,ANNOT_INST
2,ANNOTS,ANNOT_INST_T1_T2


In [54]:
# destrat out.db +ANNOTS -r ANNOT INST T -v START_HMS STOP_HMS START STOP
tbl = p.table( 'ANNOTS' , 'ANNOT_INST_T1_T2' )[[ 'ANNOT' , 'START_HMS' , 'STOP_HMS' , 'START' , 'STOP' ]]
tbl

,ANNOT,START_HMS,STOP_HMS,START,STOP
0,N1,22:06:30,22:07:00,390.0,420.0
1,N1,22:07:00,22:07:30,420.0,450.0
2,N1,22:07:30,22:08:00,450.0,480.0
3,N1,22:08:00,22:08:30,480.0,510.0
4,N1,22:08:30,22:09:00,510.0,540.0
...,...,...,...,...,...
872,W,05:23:00,05:23:30,26580.0,26610.0
873,W,05:23:30,05:24:00,26610.0,26640.0
874,W,05:24:00,05:24:30,26640.0,26670.0
875,W,05:24:30,05:25:00,26670.0,26700.0


To sort the table by start time of each annotation (rather than the default to first sort by annotation class):

In [55]:
tbl.sort_values(by='START')

,ANNOT,START_HMS,STOP_HMS,START,STOP
760,W,22:00:00,22:00:30,0.0,30.0
761,W,22:00:30,22:01:00,30.0,60.0
762,W,22:01:00,22:01:30,60.0,90.0
763,W,22:01:30,22:02:00,90.0,120.0
764,W,22:02:00,22:02:30,120.0,150.0
...,...,...,...,...,...
872,W,05:23:00,05:23:30,26580.0,26610.0
873,W,05:23:30,05:24:00,26610.0,26640.0
874,W,05:24:00,05:24:30,26640.0,26670.0
875,W,05:24:30,05:25:00,26670.0,26700.0


We see that staging annotations align at 0/30 seconds past each minute, relative to the EDF start time.    

In contrast, by default, then epoch will be aligned _within each segment_ (which here in a gapped EDF+D starts _after_ the EDF start time:

In [46]:
# luna harm1.lst -o out.db id=M10 -s EPOCH verbose
p.eval( 'EPOCH verbose' )

 ..................................................................
 CMD #1: EPOCH
   options: sig=* verbose
  set epochs, length 30 (step 30, offset 0), 890 epochs


,Command,Strata
0,EPOCH,BL
1,EPOCH,E


In [48]:
# destrat out.db +EPOCH -r E -v HMS START STOP
p.table( 'EPOCH' , 'E' )[[ 'HMS', 'START', 'STOP' ]] 

,HMS,START,STOP
0,22:00:22.300,22.3,52.3
1,22:00:52.300,52.3,82.3
2,22:01:22.300,82.3,112.3
3,22:01:52.300,112.3,142.3
4,22:02:22.300,142.3,172.3
...,...,...,...
885,05:30:18.557,27018.557,27048.557
886,05:30:48.557,27048.557,27078.557
887,05:31:18.557,27078.557,27108.557
888,05:31:48.557,27108.557,27138.557


That is, as the first _segment_ starts at `22:00:22.3`, so does the first epoch (i.e. and so stage annotations will span multiple epochs).    As noted, we can change this by forcing epochs to be aligned to staging annotations within each segment:

In [49]:
# luna harm1.lst -o out.db id=M10 -s EPOCH align verbose
p.eval( 'EPOCH align verbose' )
# destrat out.db +EPOCH -r E -v HMS START STOP
p.table( 'EPOCH' , 'E' )[[ 'HMS', 'START', 'STOP' ]] 

 ..................................................................
 CMD #1: EPOCH
   options: align sig=* verbose
 epoch definitions have changed: original epoch mappings will be lost
  within each segment, aligning epochs to 891 possible starting points from (N1,N2,N3,R,W,?,L,U,M)
  set epochs, length 30 (step 30, offset 0), 890 epochs


,HMS,START,STOP
0,22:00:30.000,30.0,60.0
1,22:01:00.000,60.0,90.0
2,22:01:30.000,90.0,120.0
3,22:02:00.000,120.0,150.0
4,22:02:30.000,150.0,180.0
...,...,...,...
885,05:30:30.000,27030.0,27060.0
886,05:31:00.000,27060.0,27090.0
887,05:31:30.000,27090.0,27120.0
888,05:32:00.000,27120.0,27150.0


That is, rather than start 22.3 seconds past the EDF start time, the first epoch is shifted to start at 30 seconds past the EDF, which aligns with the first stage annotation that starts within a non-gapped region of the recording.

The bottom line: we want to re-run `HYPNO` but first add `EPOCH align` to ensure that stages and epochs align.  Doing this for the three individuals with gapped EDF+Ds we should now see no _conflicts_:

In [81]:
# luna harm1.lst -o out.db id=F01,F10,M10 -s 'EPOCH align & HYPNO'
# the `id` variable should still be set to these three individuals from the earlier cells
res = proj.silent_proc( 'EPOCH align & HYPNO' )

In [82]:
# destrat out.db +HYPNO -v CONF
proj.table( 'HYPNO' )[[ 'ID' , 'CONF' ]] 

,ID,CONF
0,F01,0
1,F10,0
2,M10,0


We can now view hypnogram statistics, such as stage durations:

In [83]:
# destrat out.db +HYPNO -r SS/N1,N2,N3,R,W -v MINS
tbl = proj.table( 'HYPNO' , 'SS' )[[ 'ID', 'SS', 'MINS' ]] 
tbl = tbl[ tbl.SS.isin( [ 'N1', 'N2', 'N3', 'R', 'W' ] ) ]
tbl.sort_values( 'ID' )

,ID,SS,MINS
6,F01,N1,0.0
9,F01,N2,180.5
12,F01,N3,0.0
21,F01,R,0.0
27,F01,W,0.0
7,F10,N1,40.5
10,F10,N2,213.0
13,F10,N3,42.0
22,F10,R,73.5
28,F10,W,91.5


As noted in the walk-through, for `F01` we see only N2 epochs -- naturally this would be unusal / implausible in a real whole-night sleep recording. 

## EDF-MINUS

Finally, as noted in the walk-through, we can use `EDF-MINUS` to generate ungapped EDFs from EDF+D.  We'll make a `tmp/` folder in the root/parent folder:

In [85]:
import os
os.makedirs( '../tmp' )

For `F10`, we'll use the _splice_ policy (see [here](https://zzz.bwh.harvard.edu/luna-walkthrough/p2/edfd/#edf-minus) for details):

In [86]:
# luna harm1.lst id=F10 -o out.db -s EDF-MINUS policy=splice out=tmp/F10b
p = proj.inst( 'F10' )
p.eval( 'EDF-MINUS policy=splice out=../tmp/F10b' )

___________________________________________________________________
Processing: F10 | ../work/harm1/F10.edf
 duration 07.40.46, 27646s | time 22.00.00 - 05.46.23 | date 01.01.85

 signals: 58 (of 58) selected in an EDF+D file
  Fp1 | Fp2 | AF3 | AF4 | F7 | F5 | F3 | F1
  F2 | F4 | F6 | F8 | FT7 | FC5 | FC3 | FC1
  FC2 | FC4 | FC6 | FT8 | T7 | C5 | C3 | C1
  C2 | C4 | C6 | T8 | TP7 | CP5 | CP3 | CP1
  CP2 | CP4 | CP6 | TP8 | P7 | P5 | P3 | P1
  P2 | P4 | P6 | P8 | PO3 | PO4 | O1 | O2
  AFZ | FZ | FCZ | CZ | CPZ | PZ | POz | OZ
  FPZ | EDF Annotations
  extracting 'EDF Annotations' track from EDF+
 ..................................................................
 CMD #1: EDF-MINUS
   options: out=../tmp/F10b policy=splice sig=*

  settings:
     join-policy (policy)                   = splice
     retained segments (segments)           = all
     maximum sample rate allowed (max-sr)   = 1024 Hz
     segment alignment annotations (align)  = ?,N1,N2,N3,R,W
       alignment duration unit 

,Command,Strata
0,EDF_MINUS,ANNOT_SEG
1,EDF_MINUS,BL
2,EDF_MINUS,SEG


We can see some of the saved outputs (also in the console log above):

In [87]:
# destrat out.db +EDF-MINUS -r SEG
p.table( 'EDF_MINUS' , 'SEG' )

,ID,SEG,DUR_EDIT,DUR_ORIG,EDIT,INCLUDED,ORIG
0,F10,1.0,300.0,301.0,0.00->300.00,1,0.00->301.00
1,F10,2.0,9690.0,9700.0,311.20->10001.20,1,311.20->10011.20
2,F10,3.0,17640.0,17645.0,10338.60->27978.60,1,10338.60->27983.60


More detailed information, e.g. on the number of alignment annotations observed in each segment is also stored here:

In [88]:
# destrat out.db +EDF-MINUS -r SEG ANNOT
p.table( 'EDF_MINUS' , 'ANNOT_SEG' )

,ID,ANNOT,SEG,N_ALL,N_REQ
0,F10,?,1.0,0,0
1,F10,?,2.0,0,0
2,F10,?,3.0,0,0
3,F10,N1,1.0,3,3
4,F10,N1,2.0,43,43
5,F10,N1,3.0,35,35
6,F10,N2,1.0,0,0
7,F10,N2,2.0,170,170
8,F10,N2,3.0,256,256
9,F10,N3,1.0,0,0


---

The `SPANNING` command gives information on the extent to which EDF records are spanned by one or more of a set of annotations (typically, but not necessarily, stage annotations). Applying this to the newly created `F10b.edf` and `F10b.annot`:

In [90]:
os.listdir( '../tmp/' )

['F10b.edf', 'F10b.annot']

As `F10b` does not exist in the attached sample list, this will create an empty instance, and we can manually attach the EDF and annotation file:

In [91]:
p = proj.inst( 'F10b' )

In [92]:
p.attach_edf( '../tmp/F10b.edf' )

___________________________________________________________________
Processing: F10b | ../tmp/F10b.edf
 duration 07.40.30, 27630s | time 22.00.00 - 05.40.30 | date 01.01.85

 signals: 57 (of 57) selected in a standard EDF file
  AF3 | AF4 | AFZ | C1 | C2 | C3 | C4 | C5
  C6 | CP1 | CP2 | CP3 | CP4 | CP5 | CP6 | CPZ
  CZ | F1 | F2 | F3 | F4 | F5 | F6 | F7
  F8 | FC1 | FC2 | FC3 | FC4 | FC5 | FC6 | FCZ
  FPZ | FT7 | FT8 | FZ | Fp1 | Fp2 | O1 | O2
  OZ | P1 | P2 | P3 | P4 | P5 | P6 | P7
  P8 | PO3 | PO4 | POz | PZ | T7 | T8 | TP7
  TP8


True

In [93]:
p.attach_annot( '../tmp/F10b.annot' )

True

In [96]:
# luna tmp/F10b.edf annot-file=tmp/F10b.annot -o out.db -s SPANNING annot=N1,N2,N3,R,W
p.eval( 'SPANNING annot=N1,N2,N3,R,W' )

 ..................................................................
 CMD #1: SPANNING
   options: annot=N1,N2,N3,R,W sig=*


,Command,Strata
0,SPANNING,BL


In [100]:
# destrat out.db +SPANNING | behead
p.table( 'SPANNING' ).T

,0
ID,F10b
ANNOT_HMS,07:40:30.000
ANNOT_N,921
ANNOT_OVERLAP,NO
ANNOT_SEC,27630.0
INVALID_N,0
INVALID_SEC,0.0
NSEGS,1
REC_HMS,07:40:30.000
REC_SEC,27630.0


That is, we now have a single segment (`NSEGS`) with 27630 seconds of annotated signal, which spans 100% of the avaialble signal.  In contrast, the original (EDF+D) has 3 segments, with the same duration of annotated signal, but a small percentage (16 seconds) or recording not spanned by an annotation, as we'd noted above:

In [101]:
# luna harm1.lst id=F10 -o out.db -s SPANNING annot=N1,N2,N3,R,W
p = proj.inst( 'F10' )
p.eval( 'SPANNING annot=N1,N2,N3,R,W' )
# destrat out.db +SPANNING | behead
p.table( 'SPANNING' ).T

___________________________________________________________________
Processing: F10 | ../work/harm1/F10.edf
 duration 07.40.46, 27646s | time 22.00.00 - 05.46.23 | date 01.01.85

 signals: 58 (of 58) selected in an EDF+D file
  Fp1 | Fp2 | AF3 | AF4 | F7 | F5 | F3 | F1
  F2 | F4 | F6 | F8 | FT7 | FC5 | FC3 | FC1
  FC2 | FC4 | FC6 | FT8 | T7 | C5 | C3 | C1
  C2 | C4 | C6 | T8 | TP7 | CP5 | CP3 | CP1
  CP2 | CP4 | CP6 | TP8 | P7 | P5 | P3 | P1
  P2 | P4 | P6 | P8 | PO3 | PO4 | O1 | O2
  AFZ | FZ | FCZ | CZ | CPZ | PZ | POz | OZ
  FPZ | EDF Annotations
  extracting 'EDF Annotations' track from EDF+
 ..................................................................
 CMD #1: SPANNING
   options: annot=N1,N2,N3,R,W sig=*


,0
ID,F10
ANNOT_HMS,07:40:30.000
ANNOT_N,921
ANNOT_OVERLAP,NO
ANNOT_SEC,27630.0
INVALID_N,0
INVALID_SEC,0.0
NSEGS,3
REC_HMS,07:40:46.000
REC_SEC,27646.0


As above, both are valid files but the former can be simpler to work with for aforementioned reasons.

Turning to `M10`, here we'll adopt a zero-padding policy, i.e. to fill the gaps in the signals rather than splice them out and change the annotations:

In [103]:
# luna harm1.lst id=M10 -o out.db -s EDF-MINUS policy=zero-pad out=tmp/M10b
p = proj.inst( 'M10' )
p.eval( 'EDF-MINUS policy=zero-pad out=../tmp/M10b' )

___________________________________________________________________
Processing: M10 | ../work/harm1/M10.edf
 duration 07.25.36, 26736s | time 22.00.00 - 05.33.02 | date 01.01.85

 signals: 58 (of 58) selected in an EDF+D file
  Fp1 | Fp2 | AF3 | AF4 | F7 | F5 | F3 | F1
  F2 | F4 | F6 | F8 | FT7 | FC5 | FC3 | FC1
  FC2 | FC4 | FC6 | FT8 | T7 | C5 | C3 | C1
  C2 | C4 | C6 | T8 | TP7 | CP5 | CP3 | CP1
  CP2 | CP4 | CP6 | TP8 | P7 | P5 | P3 | P1
  P2 | P4 | P6 | P8 | PO3 | PO4 | O1 | O2
  AFZ | FZ | FCZ | CZ | CPZ | PZ | POz | OZ
  FPZ | EDF Annotations
  extracting 'EDF Annotations' track from EDF+
 ..................................................................
 CMD #1: EDF-MINUS
   options: out=../tmp/M10b policy=zero-pad sig=*

  settings:
     join-policy (policy)                   = zero-pad
     retained segments (segments)           = all
     maximum sample rate allowed (max-sr)   = 1024 Hz
     segment alignment annotations (align)  = ?,N1,N2,N3,R,W
       alignment duration u

,Command,Strata
0,EDF_MINUS,ANNOT_SEG
1,EDF_MINUS,BL
2,EDF_MINUS,SEG


In [104]:
# destrat out.db +EDF-MINUS -r SEG
p.table( 'EDF_MINUS' , 'SEG' )

,ID,SEG,DUR_EDIT,DUR_ORIG,EDIT,INCLUDED,ORIG
0,M10,1.0,510.0,532.0,30.00->540.00,1,22.30->554.30
1,M10,2.0,26190.0,26204.0,990.00->27180.00,1,978.56->27182.56


In [105]:
# destrat out.db +EDF-MINUS -r SEG ANNOT
p.table( 'EDF_MINUS' , 'ANNOT_SEG' )

,ID,ANNOT,SEG,N_ALL,N_REQ
0,M10,?,1.0,0,0
1,M10,?,2.0,0,0
2,M10,N1,1.0,5,5
3,M10,N1,2.0,162,162
4,M10,N2,1.0,1,0
5,M10,N2,2.0,399,399
6,M10,N3,1.0,0,0
7,M10,N3,2.0,80,80
8,M10,R,1.0,0,0
9,M10,R,2.0,114,114


See the [original walk-through](https://zzz.bwh.harvard.edu/luna-walkthrough/p2/edfd/#edf-minus) for pointers on how to use `scope()` to look at the new EDFs.

---

We'll now move on to the [next section](02_dupes.ipynb) to find likely duplicate signals.